# Week 6: Impact and Multi-Referrer Process Design

## tl;dr

This notebook refreshes the shared referral-attribution pipeline and presents the Week 6 decision outputs. It does not reimplement matching rules in notebook cells.

The default JSON contract preserves the accepted Week 5 baseline: 12 eligible bookings, 8 strict honored bookings, and 4 primary review candidates. Configured CRM-ID and unit-number signals remain visible as a separate data-quality sensitivity and do not change the primary funnel unless a data owner explicitly enables them. The recommendation is **P-MANUAL**: show an owner candidate for one distinct prior referrer and require manual review for two or more.


## Context & Methods

### Key Assumptions

- Primary identity remains exact `client + lead_id`.
- A referral must be time-valid before the booking; the latest time-valid referral is a detection reference, not ownership proof.
- Strict honored means the recorded booking source is `REFERRAL`.
- Agreement value is transaction context only. It is not commission, loss, or recoverable impact.
- Reverse-time source disagreements are Data Integrity records and never enter the primary review rate.


### 1. Refresh Shared Pipeline

In [1]:
# Locate the project root from the current notebook location.
# -> Import the reusable package.
# -> Refresh all derived outputs through the documented entry point.
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        path for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "config" / "attribution_rules.json").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Cannot locate the project root from {CURRENT_DIR}. "
        "Run this notebook from the repository root or code folder."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from referral_attribution.pipeline import run_pipeline
from referral_attribution.process_rules import build_warning_decision_table

result = run_pipeline(
    project_root=PROJECT_ROOT,
    write_outputs=True,
    verbose=False,
)
tables = result["tables"]
rules = result["rules"]

print(f"Project root detected: {PROJECT_ROOT.name}")
print("Pipeline completed and Week 6 derived outputs were refreshed.")

Project root detected: Terra Softech_2
Pipeline completed and Week 6 derived outputs were refreshed.


### 2. Impact Framing

In [2]:
# Read the output table rather than recomputing metrics in the notebook.
# -> Keep the denominator and interpretation beside every metric.
impact = tables["impact_framing_summary.csv"]
impact_display = impact.loc[
    impact["metric_key"].isin(
        [
            "filtered_bookings",
            "identity_overlap_bookings",
            "eligible_prior_referral_bookings",
            "strict_honored_rate",
            "primary_review_rate",
            "primary_review_per_1000_filtered_bookings",
            "tier_b_primary_cases",
            "tier_c_identity_cases",
            "reverse_time_source_disagreements",
        ]
    ),
    ["metric", "value", "numerator", "denominator", "unit", "decision_use", "not_a_claim"],
]
display(impact_display)

metric_values = impact.set_index("metric_key")["value"]
filtered_bookings = int(metric_values["filtered_bookings"])
eligible_bookings = int(metric_values["eligible_prior_referral_bookings"])
review_cases = int(round(metric_values["primary_review_rate"] * eligible_bookings))
review_per_thousand = metric_values["primary_review_per_1000_filtered_bookings"]
conditional_rate = metric_values["primary_review_rate"]
summary = (
    f"**Impact summary:** The filtered production-like base contains "
    f"**{filtered_bookings:,} bookings**. There are **{review_cases} primary review cases**, "
    f"equal to **{review_per_thousand:.2f} cases per 1,000 filtered bookings**. "
    f"The conditional review rate is **{conditional_rate:.1%}** among eligible prior-referral "
    f"bookings; it is a diagnostic, not a system-wide bypass rate."
)
display(Markdown(summary))

,metric,value,numerator,denominator,unit,decision_use,not_a_claim
0,Filtered production-like bookings,4713.000000,NaN,NaN,bookings,Base for operational workload context.,Not a count of referral-attributable bookings.
1,Identity-overlap bookings,4062.000000,4062.0,4713.0,bookings,Shows where the time-order gate has evidence t...,Not a conversion or bypass rate.
2,Eligible prior-referral bookings,12.000000,12.0,4062.0,bookings,Conditional cohort for strict honored and prim...,Not the total booking population.
3,Strict honored rate among eligible bookings,0.666667,8.0,12.0,rate,Checks recorded attribution inside the eligibl...,Does not prove commission payment or ownership.
4,Primary review rate among eligible bookings,0.333333,4.0,12.0,rate,Conditional attribution-quality diagnostic.,Not a system-wide bypass rate.
5,"Primary review cases per 1,000 filtered bookings",0.848716,4.0,4713.0,"cases per 1,000 bookings",Headline estimate of routine Ops review workload.,Not a commission-loss or fraud rate.
6,Tier B primary review cases,4.000000,4.0,4.0,cases,Shows ownership ambiguity that requires manual...,Not confirmed bypass or fraud.
7,Tier C identity cases,3.000000,NaN,NaN,cases,Separate identity-confirmation queue.,Does not change the primary review count.
8,Reverse-time source disagreements,28.000000,NaN,NaN,records,Data Integrity backlog for chronology repair.,Never primary bypass evidence or unpaid commis...


**Impact summary:** The filtered production-like base contains **4,713 bookings**. There are **4 primary review cases**, equal to **0.85 cases per 1,000 filtered bookings**. The conditional review rate is **33.3%** among eligible prior-referral bookings; it is a diagnostic, not a system-wide bypass rate.

### 3. Tier B Agreement Context

The value-sanity flag is deliberately separate from case eligibility. A suspicious value requires source verification, but it cannot erase an attribution review record or turn an agreement value into a commission estimate.


In [3]:
# Display one compact row per Tier B case.
# -> Preserve attribution evidence.
# -> Flag only the value context that needs verification.
tier_b_context = tables["tier_b_agreement_context.csv"]
display(tier_b_context)

,booking_id,booking_timeline_at,booking_source_norm,lead_id,referral_enquiry_id,referral_submitted_at,referrer_user_id,referrer_name,prior_referral_count,prior_referrer_count,days_between_referral_and_booking,same_project,timeline_ambiguity_flag,agreement_value,value_sanity_flag,review_reason,agreement_value_context
0,1206,2024-06-22 23:00:00,<NULL>,897,1784,2024-06-20 16:58:51.247138,541,Shivaleela,10,10,2.250796,True,False,50000.0,Positive contextual value,Multiple prior referrers,"Transaction context only; not commission, loss..."
11,935,2024-09-05 00:00:00,CHANNEL_PARTNER,887,3608,2024-09-02 15:27:01.683765,914,Fatima,61,61,2.356231,True,True,11111111.0,Repeated-digit pattern,Multiple prior referrers and timeline ambiguity,"Transaction context only; not commission, loss..."
8,3331,2025-04-03 00:00:00,CHANNEL_PARTNER,898,5989,2025-04-01 15:15:42.553829,1497,Nasser,134,134,1.364091,True,False,5000000.0,Positive contextual value,Multiple prior referrers,"Transaction context only; not commission, loss..."
10,4913,2026-03-15 00:00:00,CHANNEL_PARTNER,901,9142,2026-03-11 12:41:57.766398,1930,nitya,236,236,3.470859,True,False,2000000.0,Positive contextual value,Multiple prior referrers,"Transaction context only; not commission, loss..."


### 4. Multi-Referrer Policy Sensitivity

The table compares possible ownership rules on the same eligible bookings. It is a process simulation: it shows operational workload and unsafe auto-assignment exposure, not commission allocation or financial impact.


In [4]:
# Compare configured ownership policies in the unlimited batch view.
# -> P-MANUAL is the recommended provisional policy.
policy_summary = tables["multi_referrer_policy_sensitivity.csv"]
display(
    policy_summary[
        [
            "policy_id",
            "policy_label",
            "eligible_bookings",
            "auto_assignable_count",
            "manual_review_count",
            "tier_b_auto_owner_count",
            "tier_b_manual_review_count",
            "recommended_policy",
        ]
    ]
)

# Show the case-level outcome only for the recommended policy.
case_matrix = tables["multi_referrer_policy_case_matrix.csv"]
recommended_policy_id = rules["process"]["multi_referrer"]["recommended_policy_id"]
display(
    case_matrix.loc[
        case_matrix["policy_id"].eq(recommended_policy_id),
        [
            "booking_id",
            "prior_referral_count",
            "prior_referrer_count",
            "latest_days_between_referral_and_booking",
            "same_project_prior_referral_available",
            "owner_result",
            "policy_decision",
        ],
    ]
)

,policy_id,policy_label,eligible_bookings,auto_assignable_count,manual_review_count,tier_b_auto_owner_count,tier_b_manual_review_count,recommended_policy
0,P-LAST,Latest prior referrer owns,12,12,0,4,0,False
1,P-FIRST,First prior referrer owns,12,12,0,4,0,False
2,P-MANUAL,Manual review for multiple prior referrers,12,7,5,0,4,True
3,P-SAME-PROJECT-LAST,Latest same-project referrer owns,12,12,0,4,0,False


,booking_id,prior_referral_count,prior_referrer_count,latest_days_between_referral_and_booking,same_project_prior_referral_available,owner_result,policy_decision
24,1206,10,10,5.153140,True,MANUAL_REVIEW,Manual ownership review
25,1747,1,1,15.499401,True,319,Auto assign latest prior referrer
26,1781,1,1,61.425285,True,319,Auto assign latest prior referrer
27,1999,1,1,14.502770,True,319,Auto assign latest prior referrer
28,2310,1,1,17.370771,True,453,Auto assign latest prior referrer
29,2436,1,1,0.340636,True,1085,Auto assign latest prior referrer
30,2484,2,1,71.539278,True,319,Auto assign latest prior referrer
31,2523,2,1,75.539278,True,319,Auto assign latest prior referrer
32,3331,134,134,289.194796,True,MANUAL_REVIEW,Manual ownership review
33,4785,231,231,609.194967,True,MANUAL_REVIEW,Manual ownership review


### 5. Product Warning Rule and Window Sensitivity

In [5]:
# Render the configured Product decision table.
# -> Keep the 30-day user-interface rule separate from batch sensitivity.
warning_table = build_warning_decision_table(rules)
display(
    warning_table[
        [
            "scenario",
            "warning_severity",
            "action",
            "override_reason_required",
            "manual_review_required",
            "decision",
        ]
    ]
)

process_sensitivity = tables["process_window_and_policy_sensitivity.csv"]
display(
    process_sensitivity.loc[
        process_sensitivity["policy_id"].eq(recommended_policy_id),
        [
            "attribution_window",
            "eligible_bookings",
            "primary_review_candidates",
            "auto_assignable_count",
            "manual_review_count",
            "tier_b_manual_review_count",
        ],
    ]
)

,scenario,warning_severity,action,override_reason_required,manual_review_required,decision
0,No prior referral,None,No warning,False,False,Proceed
1,One same-project referrer,Medium,Soft warning + show owner candidate,False,False,Proceed with warning
2,Multiple same-project referrers,High,Soft warning + required override reason + manu...,True,True,Escalate
3,Cross-project referrals only,Low,Low-severity warning + event log; no automatic...,False,False,Proceed and log


,attribution_window,eligible_bookings,primary_review_candidates,auto_assignable_count,manual_review_count,tier_b_manual_review_count
2,30 days,11,4,7,4,3
6,60 days,11,4,6,5,4
10,90 days,12,4,7,5,4
14,180 days,12,4,7,5,4
18,Unlimited,12,4,7,5,4


## Takeaways

1. Use **0.85 primary review cases per 1,000 filtered bookings** as the routine Ops workload metric, and retain the conditional review rate for attribution-quality diagnosis.
2. Use **P-MANUAL** for multi-referrer history. Latest referral remains useful for detection and display, but not as an automatic ownership or payout decision when multiple people referred the same lead.
3. Pilot a **30-day soft warning** for non-referral booking saves. Measure warning completion, override reasons, queue age, booking friction, and future strict-honored and reverse-time rates before considering a hard stop.
4. Keep the four primary review candidates, three Tier C identity checks, and 28 reverse-time records in separate queues. The supporting docs state what the data can and cannot prove.


## Reproducibility

- Pipeline command: `python3 -m referral_attribution.pipeline`
- Test command: `python3 -m pytest`
- Rule source: `config/attribution_rules.json`
- Supporting documents: `log/decision_memo.md`, `log/multi_referrer_policy_brief.md`, and `log/process_rule_spec.md`
